# The connection between a 2-mode coupler and a polarisation system

In this notebook we use our analytic solution to the coherent coupler in the Weierstrass notation to explore coordinate transformations. We show that it is possible to transform the coherent coupler to a canonical coordinate system in which the solutions are single valued functions, namely Kronecker theta functions. This transformation is from the coherent coupler system to nonlinear polarisation dynamics.

In [3]:
from sympy import *
from time import time
from itertools import combinations_with_replacement

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi, varsigma) = symbols(
    '''delta, nu, Aeff, chi, varsigma'''
)

gbar2 = Symbol('gbar2', latex_name=r'\bar{g_2}')
gbar3 = Symbol('gbar3', latex_name=r'\bar{g_3}')
gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

Det = Function('Det') # Unevaluated determinant

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
rhohat = Function('rhohat', latex_name=r'\hat{rho}')

kappa = Function('kappa')
phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

A = Function('A')
Ac = Function('Ac')
B = Function('B')
Bc = Function('Bc')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammahat = IndexedBase('gammahat', latex_name=r'\hat{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lamda = IndexedBase('lamda') # special sympy spelling
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

## Elliptic function identities

Period convention: choose primitive half-periods with $\operatorname{Im}(\omega_3/\omega_1)>0$, and $\eta_j=\zeta(\omega_j)$. Then $\eta_1\omega_3-\eta_3\omega_1=i\pi/2$. Quasi-period constants are linear combinations of $\eta_1,\eta_3$; they are not zeta evaluated at those lattice combinations. The historical numerical helper is unchanged.

In [ ]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)

quasi_period_sigma_b = Eq(
    sigma(2*m*omega[3] + 2*n*omega[1] + z, g2, g3),
    (-1)**(m*n + m + n)*sigma(z, g2, g3)*
    exp(2*(m*omega[3] + n*omega[1] + z)*(m*zeta(omega[3], g2, g3) + n*zeta(omega[1], g2, g3)))
)

# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_sigma_dbl_ratio
quasi_period_sigma_b

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

## The solution in original coordinates

We start by reminding ourselves of the solutions derived in *The general 2 mode case of the coherent coupler* notebook for modes in the original coordinates $u,v$:

In [ ]:
Lams = [
    Eq(
        Lamda[0, m],
        -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
        + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
    ),
    Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))
]

u_sqrt_wp_z0_z1 = Eq(
    u(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])
)
v_sqrt_wp_z0_z1 = Eq(
    v(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])
)

uv_wp_z0z1muj = Eq(
    u(z, mu[j])*v(z, mu[j]),
    (wp(z - z0, g2, g3) - wp(-z0 + mu[j], g2, g3))
    *wpp(z1, g2, g3)/
    ((-wp(z1, g2, g3) + wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sqrt(d[4]))
)

uv_wp_mu1_mu2_const = Eq(
    u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2]),
    (wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*wpp(z1, g2, g3)/(
        (-wp(z1, g2, g3) + wp(-z0 + mu[1], g2, g3))*(-wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3))*sqrt(d[4])
    )
)

sum_r1j_1 = Eq(Sum(r[1, j], (j, 1, 2)), 1)
r1j_log_part = Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1
uv_wp_mu1_mu2_const
uv_wp_z0z1muj
sum_r1j_1
r1j_log_part

for _ in Lams:
    _

### The original parameters

In [ ]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]

g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _

g2_dj
g3_dj

In [ ]:
B_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(gamma[j] - lamda[1])*sqrt(d[4])
)
C_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)/(gamma[j] - lamda[1])
)
D_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)*wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2,
    (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*sqrt(d[4])
)


wp_z1_lam_uv_j = Eq(
    wpp(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(d[4])),
    -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2
)
wp_z1_lam_uv_j_sqrt = Eq(
    sqrt(wpp(z1, g2, g3))/sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))/sqrt(sqrt(d[4])),
    I*varsigma*sqrt(lamda[1] + Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2)
)
d4_lam_0 = Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)
d5_dj = Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)
wppz1_djs = Eq( 4*wpp(z1, g2, g3)/sqrt(d[4]), -d[5])

wp_z0_mu_j_d = Eq(
    wp(-z0 + mu[j], g2, g3),
    d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - 
    (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1]))
)
wpp_z0_mu_j_d = Eq(
    wpp(-z0 + mu[j], g2, g3),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)
    *(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2)
)
wp_z1_z0_mu_j_d5_lam = Eq(
    -(1/B_wpz1_z0_muj_to_d_lam1_gamj.lhs)*wppz1_djs.lhs*sqrt(d[4])/4, 
    -(1/B_wpz1_z0_muj_to_d_lam1_gamj.rhs)*wppz1_djs.rhs*sqrt(d[4])/4
)

wppz1_djs_sqrt = Eq(
    sqrt(fraction(wppz1_djs.lhs)[0])/sqrt(fraction(wppz1_djs.lhs)[1]), 
    I*varsigma*sqrt(-wppz1_djs.rhs)
)



B_wpz1_z0_muj_to_d_lam1_gamj
C_wpz1_z0_muj_to_d_lam1_gamj
D_wpz1_z0_muj_to_d_lam1_gamj
wp_z1_z0_mu_j_d5_lam
wp_z1_lam_uv_j
wp_z1_lam_uv_j_sqrt
wppz1_djs
wppz1_djs_sqrt
wp_z0_mu_j_d
wpp_z0_mu_j_d

d4_lam_0
d5_dj


We recall the equations from the original derivation of the solution in terms of $\rho, b_j, \gamma_j, \lambda_1$:

In [ ]:
drhop_b = Eq(
    Derivative(rho(z), z)**2, 
    (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2))
)
drhop_d = Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

drho_2z = Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j-1)*d[j]/2, (j, 0, 4)))
drho_2z_b = Eq(
    Derivative(rho(z), (z, 2)),
    (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) + 2*(-2*rho(z) + gamma[1] + gamma[2])
)

gamma_lamda_bj = Eq(
    drhop_b.rhs.subs(rho(z), lamda[1]).doit()
    .subs(gamma[2], - gamma[1])
    .expand().collect(4*gamma[1]**2 - 4*lamda[1]**2, factor),
    0
)
gamma_lamda_bj_gam1 = Eq(gamma[1]**2, solve(gamma_lamda_bj, gamma[1]**2)[0])

drhop_b
drhop_d
drho_2z
drho_2z_b
gamma_lamda_bj_gam1

In [ ]:
bs_as_ds = [_.subs([_.args for _ in c_j_coeffs]) for _ in d_j_coeffs]

b0_d1 = Eq(b[0], solve(bs_as_ds[1], b[0])[0])
b2_d3 = Eq(b[2], solve(bs_as_ds[3], b[2])[0])
b1_sqrd_as_d = Eq(
    b[1]**2,
    solve(
        bs_as_ds[2]
        .subs([
            b0_d1.args, 
            (2*bs_as_ds[4].rhs/bs_as_ds[3].rhs, 2*bs_as_ds[4].lhs/bs_as_ds[3].lhs)
        ]),
        b[1]**2
    )[0]
)
b_to_d_subs = [
    b0_d1.args,
    b2_d3.args,
    b1_sqrd_as_d.args
]

for _ in bs_as_ds:
    _
print()
for _ in b_to_d_subs:
    Eq(*_)
    
def generate_lambda_reductions(max_power=12):
    """
    Generate substitutions reducing lamda**n (n >= 4)
    to expressions at most cubic in lamda.
    """
    subs = {}

    # base reduction: λ⁴
    subs[lamda[1]**4] = -(d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3) / d[4]

    # build higher powers
    for _n in range(5, max_power + 1):
        subs[lamda[1]**_n] = expand(lamda * subs[lamda[1]**(_n - 1)]).subs(subs)

    return subs

lamda_d_reductions = generate_lambda_reductions()

### Reorganising the original equations of motion

In [ ]:
ajk_syms = [(a[2, 1], a[1, 2])]

uv_j_rho = Eq(u(z,mu[j])*v(z,mu[j]), gamma[j] - rho(z))

duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]),z), 
    Product(v(z, mu[k]), (k, 1, 2)) - Product(u(z, mu[k]), (k, 1, 2))
)

uprodj = Eq(
    Product(u(z, mu[j]), (j, 1, 2)),
    -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + 
    Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + 
    Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2
)
vprodj = Eq(uprodj.lhs + duvj.rhs.replace(k,j), uprodj.rhs + duvj.lhs.replace(k,j))
a_0_eq = Eq(uprodj.rhs + vprodj.rhs, uprodj.lhs + vprodj.lhs)

du_phase_mod_part = (a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*u(z,mu[j])
dv_phase_mod_part = -(a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*v(z,mu[j])

du_mixing_part = Product(v(z,mu[k]), (k,1,2))/v(z, mu[j])
dv_mixing_part = -Product(u(z,mu[k]), (k,1,2))/u(z, mu[j])

duj = Eq(diff(u(z,mu[j]),z), -du_phase_mod_part + du_mixing_part)
dvj = Eq(diff(v(z,mu[j]),z), (-dv_phase_mod_part).factor() + dv_mixing_part)
duj_subs = [duj.subs(j, _j + 1).args for _j in range(2)]
dvj_subs = [dvj.subs(j, _j + 1).args for _j in range(2)]
duj_dvj_subs = duj_subs + dvj_subs


assert (
    diff(solve(a_0_eq.doit(),a[0])[0],z).subs(duj_dvj_subs).doit().subs(ajk_syms).simplify() == 0
), 'a0 not conserved'


uv_j_rho
uprodj
vprodj
a_0_eq

duj
dvj
duvj

In [ ]:
rho_integral = Eq(
    Integral(-rho(z) + rho(mu[j]),z),
    z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) -
    log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]) + C
)
zeta_z1_gamma_lam = (
    pw_to_zw_identity_one_sided
    .subs([(z,z1),(x,mu[j]-z0)])
    .subs([B_wpz1_z0_muj_to_d_lam1_gamj.args])
)
zeta_z1_gamma_lam = Eq(
    (-(zeta_z1_gamma_lam.rhs - 2*zeta(z1, g2, g3))/sqrt(d[4])),
    (-(zeta_z1_gamma_lam.lhs - 2*zeta(z1, g2, g3))/sqrt(d[4])).expand()
)

uv_integral_log_sigma = rho_integral.subs([
    (rho(z), solve(uv_j_rho, rho(z))[0]), 
    (gamma[j], rho(mu[j])),
    zeta_z1_gamma_lam.args
])

rho_integral
zeta_z1_gamma_lam
uv_integral_log_sigma

## The transfrom to polarisation like dynamics

We will now explore a nonlinear transformation that removes the square root of the $z$ dependent $\wp$ term from the denominator of the solutions. We will not yet do anything to the phase part so we anticipate solutions to be the product of a Kronecker theta function and an exponential factor with an essential singularities. There will be two modes not three, unlike the transform to degenerate FWM considered previously in relation to the coherent coupler.

In [ ]:
u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1

We thus define new modes in hat coordinates as follows:

In [ ]:
uhat_sigma_j = Eq(
    uhat(z, mu[j]), 
    sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])
)
vhat_sigma_j = Eq(
    vhat(z, mu[j]), 
    sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])
)
uvhat_wp_j = Eq(
    uhat_sigma_j.lhs*vhat_sigma_j.lhs, 
    (uhat_sigma_j.rhs*vhat_sigma_j.rhs)
    .subs([
        (
            sigma_p_identity.subs([(x,z-z0), (y, mu[j] - z0)]).rhs,
            sigma_p_identity.subs([(x,z-z0), (y, mu[j] - z0)]).lhs
        )
    ])
    .simplify()
)
uv_j_d5_gam_wp_z1 = Eq(
    uvhat_wp_j.lhs + wp_z1_z0_mu_j_d5_lam.rhs,
    uvhat_wp_j.rhs + wp_z1_z0_mu_j_d5_lam.lhs
)
uv_j_to_uvhat_j = uv_wp_z0z1muj.subs([
    (uvhat_wp_j.rhs, uvhat_wp_j.lhs),
    (uv_j_d5_gam_wp_z1.rhs, uv_j_d5_gam_wp_z1.lhs),
    wp_z1_z0_mu_j_d5_lam.args,
    wppz1_djs.args
])
uv_12_d5_gam_wp_z1 = Eq(
    (uv_j_d5_gam_wp_z1.lhs*(gamma[j] - lamda[1])).expand().collect(d[5], factor),
    uv_j_d5_gam_wp_z1.rhs*(gamma[j] - lamda[1])
)
uv_12_d5_gam_wp_z1 = Eq(
    uv_12_d5_gam_wp_z1.lhs.subs(j,1) + uv_12_d5_gam_wp_z1.lhs.subs(j,2),
    (uv_12_d5_gam_wp_z1.rhs.subs(j,1) + uv_12_d5_gam_wp_z1.rhs.subs(j,2))
    .expand().subs(gamma[2], - gamma[1]).factor()
)

uvhat12_to_uvhatj = uv_12_d5_gam_wp_z1.subs(uv_j_d5_gam_wp_z1.rhs, uv_j_d5_gam_wp_z1.lhs)
uvhat12_to_uvhatj = Eq(
    -uvhat12_to_uvhatj.rhs/(2*lamda[1]),
    -uvhat12_to_uvhatj.lhs/(2*lamda[1])
)


uvhat_integral_log_sigma = uv_integral_log_sigma.replace(*uv_j_to_uvhat_j.args)
uvhat_integral_log_sigma = Eq(
    sqrt(d[4])*(
        uvhat_integral_log_sigma.lhs 
        - uvhat_integral_log_sigma.lhs 
        - uvhat_integral_log_sigma.rhs.args[2]
    ),
    sqrt(d[4])*(
        uvhat_integral_log_sigma.rhs 
        - uvhat_integral_log_sigma.lhs 
        - uvhat_integral_log_sigma.rhs.args[2]
    )
)
uvhat_integral_phiz = Eq(uvhat_integral_log_sigma.lhs, sqrt(d[4])*Integral(phi(z),z))


uhat_sigma_j
vhat_sigma_j
uvhat_wp_j
uv_j_d5_gam_wp_z1
uv_12_d5_gam_wp_z1
uvhat12_to_uvhatj
uv_j_to_uvhat_j
uvhat_integral_log_sigma
uvhat_integral_phiz

The following modal power relations exist between coordinates and we note a Mobius transfrom exists between any modal power in one coordinate system and any modal power in the other. We introduce the auxiliary function $h(z)$.

In [ ]:
# Relating sums of mode powers in both coordinates
uvhat12_sum_to_uv12_sum = Eq(
    (1/wp_z1_lam_uv_j.lhs)
    .subs([
        (uv_12_d5_gam_wp_z1.rhs/(2*lamda[1]), uv_12_d5_gam_wp_z1.lhs/(2*lamda[1]))
    ])*wppz1_djs.lhs,
    wppz1_djs.rhs/wp_z1_lam_uv_j.rhs
)

uv_j_to_uvhat_j_sym = uv_j_to_uvhat_j.subs([(uvhat12_to_uvhatj.lhs.expand(), uvhat12_to_uvhatj.rhs)])

uv_to_uvhat_sum = Eq(
    (2*d[5]/uvhat12_sum_to_uv12_sum.rhs).doit(), 
    2*d[5]/uvhat12_sum_to_uv12_sum.lhs
)

h_u_v_hat = Eq(
    h(z),
    -(gamma[1] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1])
    - (gamma[2] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) + d[5]/2
)

h_as_u_v = Eq(
    uvhat12_sum_to_uv12_sum.lhs.subs(d[5], solve(h_u_v_hat, d[5])[0]).simplify()*lamda[1]/2,
    uvhat12_sum_to_uv12_sum.rhs*lamda[1]/2
)

uvhat_j_rho = Eq(uhat(z, mu[j])*vhat(z, mu[j]), gammahat[j] - rhohat(z))
uv_hats_to_rhohat_subs = [
    uvhat_j_rho.subs(j,1).args, 
    uvhat_j_rho.subs(j,2).args
]

gams2_to_1_subs = [
    (gamma[2], -gamma[1]),
    (gammahat[2], - gammahat[1])
]

h_as_rhohat = Eq(
    h_u_v_hat.lhs, 
    h_u_v_hat.rhs
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .expand().collect(rhohat(z), factor)
)

rhohat_as_h = Eq(rhohat(z), solve(h_as_rhohat, rhohat(z))[0])


uvhat12_sum_to_uv12_sum
uv_to_uvhat_sum
uv_j_to_uvhat_j_sym
h_u_v_hat
h_as_u_v
uvhat_j_rho
h_as_rhohat
rhohat_as_h

The modes in each coordinate system are thus related to each other as follows:

In [ ]:
# Mode transforms
_sqrt_d5_id = Eq(
    sqrt(d[5])/sqrt(d[5]/(-gamma[j]+lamda[1])),
    I*sqrt(gamma[j] - lamda[1])
)

to_hat_transform_subs =[
    (uvhat_wp_j.rhs, uvhat_wp_j.lhs),
    (uv_j_d5_gam_wp_z1.rhs, uv_j_d5_gam_wp_z1.lhs),
    wp_z1_z0_mu_j_d5_lam.args,
    wppz1_djs.args,
    (uhat_sigma_j.rhs, uhat_sigma_j.lhs),
    (vhat_sigma_j.rhs, vhat_sigma_j.lhs),
    wppz1_djs_sqrt.args,
    uvhat_integral_phiz.args,
    r1j_log_part.args,
    _sqrt_d5_id.args
]
uj_as_uhatj = u_sqrt_wp_z0_z1.subs(to_hat_transform_subs)
vj_as_vhatj = v_sqrt_wp_z0_z1.subs(to_hat_transform_subs)

u_v_j_as_hats_subs = [
    uj_as_uhatj.subs(j,1).args,
    uj_as_uhatj.subs(j,2).args,
    vj_as_vhatj.subs(j,1).args,
    vj_as_vhatj.subs(j,2).args
]

uvhat12_to_uvhatj_sqrt = Eq(
    2*sqrt(uvhat12_to_uvhatj.lhs).simplify(), 
    2*sqrt(fraction(uvhat12_to_uvhatj.rhs)[0])/sqrt(fraction(uvhat12_to_uvhatj.rhs)[1])
)
uj_as_uhatj_b = Eq(
    uj_as_uhatj.lhs, 
    uj_as_uhatj.rhs
    .simplify()
    .subs([uvhat12_to_uvhatj_sqrt.args])
)
vj_as_vhatj_b = Eq(
    vj_as_vhatj.lhs, 
    vj_as_vhatj.rhs
    .simplify()
    .subs([uvhat12_to_uvhatj_sqrt.args])
)

# Alternative version
_sqrt_d5_id_b = Eq(
    sqrt(d[5]/(-gamma[j]+lamda[1])),
    I*sqrt(d[5]/(gamma[j]-lamda[1]))
)

to_hat_transform_subs_b =[
    wp_z1_z0_mu_j_d5_lam.args,
    (uhat_sigma_j.rhs, uhat_sigma_j.lhs),
    (vhat_sigma_j.rhs, vhat_sigma_j.lhs),
    uvhat_integral_phiz.args,
    r1j_log_part.args,
    _sqrt_d5_id_b.args
]
uhatj_as_uj_b = u_sqrt_wp_z0_z1.subs(to_hat_transform_subs_b)
vhatj_as_vj_b = v_sqrt_wp_z0_z1.subs(to_hat_transform_subs_b)

uhatj_as_uj_b = Eq(
    uhat(z, mu[j]), 
    varsigma**2*solve(uhatj_as_uj_b, uhat(z, mu[j]))[0]
    .subs([(1/wp_z1_lam_uv_j_sqrt.lhs, 1/wp_z1_lam_uv_j_sqrt.rhs)])
)
vhatj_as_vj_b = Eq(
    vhat(z, mu[j]), 
    varsigma**2*solve(vhatj_as_vj_b, vhat(z, mu[j]))[0]
    .subs([(1/wp_z1_lam_uv_j_sqrt.lhs, 1/wp_z1_lam_uv_j_sqrt.rhs)])
)

u_v_j_as_hats_subs_b = [
    uj_as_uhatj_b.subs(j,1).doit().args,
    uj_as_uhatj_b.subs(j,2).doit().args,
    vj_as_vhatj_b.subs(j,1).doit().args,
    vj_as_vhatj_b.subs(j,2).doit().args
]

uj_as_uhatj_h = uj_as_uhatj_b.expand().subs([
    (
        h_u_v_hat.rhs.expand(), 
        h_u_v_hat.lhs
    )
])
vj_as_vhatj_h = vj_as_vhatj_b.expand().subs([
    (
        h_u_v_hat.rhs.expand(), 
        h_u_v_hat.lhs
    )
])

u_v_j_as_hats_h_subs = [
    uj_as_uhatj_h.subs(j,1).doit().args,
    uj_as_uhatj_h.subs(j,2).doit().args,
    vj_as_vhatj_h.subs(j,1).doit().args,
    vj_as_vhatj_h.subs(j,2).doit().args
]


uj_as_uhatj
vj_as_vhatj
print()
uj_as_uhatj_b
vj_as_vhatj_b
print()
uj_as_uhatj_h
vj_as_vhatj_h
print()
uhatj_as_uj_b
vhatj_as_vj_b

The modes in the hat coordinate system inherit intermodal power conservation laws as follows:

In [ ]:
# Intermodal power conservation laws in both coordinates

uv_to_rho_subs = [
    uv_j_rho.subs(j,1).args,
    uv_j_rho.subs(j,2).args
]

uvhat_12_int_const = Eq(
    (uv_j_to_uvhat_j.lhs.subs(j,1) - uv_j_to_uvhat_j.lhs.subs(j,2))
    .subs(uv_to_rho_subs),
    uv_j_to_uvhat_j.rhs.subs(j,1) - uv_j_to_uvhat_j.rhs.subs(j,2)
)
uvhat_12_int_const_b = Eq(
    (uvhat_12_int_const.lhs - uvhat_12_int_const.rhs).factor(),
    0
)
uvhat_12_int_const_b = Eq(
    (uvhat_12_int_const_b.lhs*fraction(uvhat_12_int_const_b.lhs)[1]
     /(d[5]*(gamma[1] - lamda[1])*(gamma[2] - lamda[1]))/4)
    .expand().collect([d[5]], factor),
    0
)
_d5_gam_eq = (d[5]/(gamma[1] - lamda[1]) - d[5]/(gamma[2] - lamda[1]))
uvhat_12_int_const_b = Eq(
    - uvhat_12_int_const_b.rhs + uvhat_12_int_const_b.lhs.subs(d[5],0),
    (- uvhat_12_int_const_b.lhs + uvhat_12_int_const_b.lhs.subs(d[5],0))
    .subs(_d5_gam_eq.factor(), _d5_gam_eq)
)

gammahat12_gamma12 = uvhat_12_int_const_b.subs(uv_hats_to_rhohat_subs)
gammahat_1_to_gamma_1 = Eq(
    gammahat12_gamma12.lhs
    .subs(gams2_to_1_subs)
    .simplify()/2,
    gammahat12_gamma12.rhs
    .subs(gams2_to_1_subs)
    .simplify()/2
)

gammhat_sum_0 = Eq(gammahat[1] + gammahat[2], 0)

rhohat_to_uv12_sum = uvhat12_sum_to_uv12_sum.subs(uv_hats_to_rhohat_subs)
rhohat_to_uv12_sum = Eq(
    rhohat_to_uv12_sum.lhs
    .subs(gams2_to_1_subs)
    .expand().collect(rhohat(z), factor)/4, 
    rhohat_to_uv12_sum.rhs/4
)


uvhat_12_int_const
uvhat_12_int_const_b
uvhat_j_rho
gammahat12_gamma12
gammahat_1_to_gamma_1
gammhat_sum_0
rhohat_to_uv12_sum

### The derivative of $h(z)$

In [ ]:
_h_factor_subs = Eq((h_u_v_hat.rhs*2).expand(), h_u_v_hat.rhs*2)

du_dv_js = [
    duj.doit().subs(j,1),
    duj.doit().subs(j,2),
    dvj.doit().subs(j,1),
    dvj.doit().subs(j,2)
]
du_dv_js_subs = [_.args for _ in du_dv_js]

assert (
    Derivative(u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2]), z)
    .doit()
    .subs(du_dv_js_subs)
    .simplify()
) == 0, 'intermodal power not conserved'


dh_as_u_v = Eq(
    diff(h_as_u_v.lhs,z), 
    diff(h_as_u_v.rhs.doit(), z)
    .subs(du_dv_js_subs)
    .factor()
)
dh_as_u_v_hat = Eq(
    dh_as_u_v.lhs,
    dh_as_u_v.rhs
    .subs([uv_to_uvhat_sum.args])
    .subs(u_v_j_as_hats_subs_b)
    .subs(varsigma**2, 1)
    .factor()
    .subs([_h_factor_subs.args])
)
dlog_h_as_u_v_hat = Eq(
    dh_as_u_v_hat.lhs/h_u_v_hat.lhs,
    (dh_as_u_v_hat.rhs/h_u_v_hat.rhs).factor()
)

drhohat = Eq(
    dh_as_u_v_hat.lhs.subs([h_as_rhohat.args]).doit()/(-2*lamda[1]),
    dh_as_u_v_hat.rhs/(-2*lamda[1])
)

for _ in du_dv_js:
    _
print()

h_as_rhohat
dh_as_u_v
dh_as_u_v_hat
dlog_h_as_u_v_hat
drhohat

### Transforming the Hamiltonian to hat coordinates

In [ ]:
_uv_hat_num = fraction(uv_j_to_uvhat_j_sym.rhs)[0]
_uv_hat_den = fraction(uv_j_to_uvhat_j_sym.rhs)[1]
_uv_j_to_uvhat_j_sym_sum_aj = Eq(
    Sum(uv_j_to_uvhat_j_sym.lhs*a[j],(j,1,2)),
    2*lamda[1]*Sum(
        (_uv_hat_num*a[j]/(2*lamda[1])).factor(),
        (j,1,2)
    )/_uv_hat_den
)
_ajk_term_sum_l = Eq(
    uv_j_to_uvhat_j_sym.lhs*uv_j_to_uvhat_j_sym.lhs.subs(j,k)*a[j,k]/2,
    uv_j_to_uvhat_j_sym.rhs*uv_j_to_uvhat_j_sym.rhs.subs(j,k)*a[j,k]/2
)
_uv_j_to_uvhat_j_sym_sum_ajk = Eq(
    Sum(_ajk_term_sum_l.lhs,(j,1,2),(k,1,2)),
    Sum(fraction(_ajk_term_sum_l.rhs)[0],(j,1,2),(k,1,2))/fraction(_ajk_term_sum_l.rhs)[1]
)

a_0_eq_hat = Eq(
    a_0_eq.lhs
    .subs([
        _uv_j_to_uvhat_j_sym_sum_aj.args, 
        _uv_j_to_uvhat_j_sym_sum_ajk.args
    ])
    ,
    a_0_eq.rhs
    .doit()
    .subs(u_v_j_as_hats_subs_b)
    .subs([(varsigma**2, 1)])
)
a_0_eq_hat = Eq(
    sum(_*_uv_hat_den**2 for _ in a_0_eq_hat.lhs.args),
    sum(_*_uv_hat_den**2 for _ in a_0_eq_hat.rhs.args)
)

# In terms of rhohat
a_0_eq_rho = (
    a_0_eq_hat
    .doit()
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
)
aj_to_bj_subs = [
    (a[2,1], a[1,2]),
    (a[1,2], solve(b_j_coeffs[2].doit().subs([(a[2,1], a[1,2]), (gamma[2], -gamma[1])]), a[1,2])[0]),
    (a[0], solve(b_j_coeffs[0].doit().subs([(a[2,1], a[1,2]), (gamma[2], -gamma[1])]), a[0])[0]),
    (a[2], solve(b_j_coeffs[1].doit().subs([(a[2,1], a[1,2]), (gamma[2], -gamma[1])]), a[2])[0])
]

a_0_eq_rho_b = Eq(
    a_0_eq_rho.lhs
    .subs(aj_to_bj_subs)
    .expand().collect(rhohat(z), factor)
    .subs(aj_to_bj_subs)
    .expand().collect(rhohat(z), factor),
    a_0_eq_rho.rhs
    .subs(aj_to_bj_subs)
    .factor()
)

# In terms of h

a_0_eq_h = Eq(
    a_0_eq_rho_b.lhs
    .subs([rhohat_as_h.args, gammahat_1_to_gamma_1.args])
    .expand().collect(h(z), factor),
    a_0_eq_rho_b.rhs
    .subs([rhohat_as_h.args, gammahat_1_to_gamma_1.args])
    .factor()
)

# One over h is a useful auxilary for transforming diffs
a_0_eq_on_over_h = Eq(
    0,
    sum(_/h(z) for _ in (a_0_eq_h.rhs - a_0_eq_h.lhs).args)
)
_ooh_coeff = a_0_eq_on_over_h.rhs.coeff(h(z),-1)
a_0_eq_on_over_h = Eq(
    -(a_0_eq_on_over_h.lhs - _ooh_coeff/h(z))/_ooh_coeff, 
    -sum(_/_ooh_coeff for _ in (a_0_eq_on_over_h.rhs - _ooh_coeff/h(z)).args)
)



a_0_eq_hat
print()
a_0_eq_rho_b
print()
a_0_eq_h
print()
a_0_eq_on_over_h

### Transforming the system of differential equations to hat coordinates

We now transform the system of differential equations to hat coordinates. We remind ourselves of the original system:

In [ ]:
for _ in du_dv_js:
    _

In [ ]:
_u_v_hats = [uhat(z, mu[1]), uhat(z, mu[2]), vhat(z, mu[1]), vhat(z, mu[2])]

d_u_v_h_hats = [
    Eq(
        diff(_uv,z),
        solve(
            _d.subs(u_v_j_as_hats_h_subs).doit().subs([(varsigma**2, 1)]), 
            diff(_uv,z)
        )[0]
        .expand().collect([diff(h(z),z), h(z)], factor)
    )
    for _uv, _d in zip(_u_v_hats, du_dv_js)
]
d_u_v_h_hats_subs = [_.args for _ in d_u_v_h_hats]

assert (
    Derivative(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), z)
    .doit()
    .subs(d_u_v_h_hats_subs)
    .subs([dlog_h_as_u_v_hat.args, (gamma[2], -gamma[1])])
    .factor()
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .factor()
) == 0, 'intermodal power not conserved'

for _ in d_u_v_h_hats:
    _

In [ ]:
_uu_h_sub = (uhat(z, mu[1])*uhat(z, mu[2]), solve(a_0_eq_h, uhat(z, mu[1])*uhat(z, mu[2]))[0])
_vv_h_sub = (vhat(z, mu[1])*vhat(z, mu[2]), solve(a_0_eq_h, vhat(z, mu[1])*vhat(z, mu[2]))[0])

_collects = [
    sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1]),
    h(z), 
    *_u_v_hats, 
    _uu_h_sub[0], 
    _vv_h_sub[1]
]
d_u_v_h_hats_b = [
    Eq(
        _d.lhs,
        _d.rhs
        .subs([dlog_h_as_u_v_hat.args, _uvh])
        .subs(uv_hats_to_rhohat_subs)
        .subs([rhohat_as_h.args])
        .subs(gams2_to_1_subs)
        .subs([gammahat_1_to_gamma_1.args])
        .subs(aj_to_bj_subs)
        .expand().collect(_collects, factor)
        .subs([a_0_eq_on_over_h.args])
        .expand().collect(_collects, factor)
    )
    for _uvh, _d in zip([_uu_h_sub, _uu_h_sub, _vv_h_sub, _vv_h_sub], d_u_v_h_hats)
]
d_u_v_h_hats_b_subs = [_.args for _ in d_u_v_h_hats_b]

assert (
    Derivative(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), z)
    .doit()
    .subs(d_u_v_h_hats_b_subs)
    .subs([dlog_h_as_u_v_hat.args, (gamma[2], -gamma[1])])
    .factor()
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .factor()
) == 0, 'intermodal power not conserved'

for _ in d_u_v_h_hats_b:
    _

### The canonical Hamiltonian for the new system

In [ ]:
# for _ in b_j_coeffs:
#     _.doit().subs(gamma[2], -gamma[1])
    
# for _ in c_j_coeffs:
#     _.doit().subs(gamma[2], -gamma[1])

# for _ in d_j_coeffs:
#     _.doit().subs(gamma[2], -gamma[1])
    
# aj_to_bj_subs

In [ ]:
three_term_products = [prod(c) for c in combinations_with_replacement(_u_v_hats, 3)]
print(len(three_term_products))

In [ ]:
_a0h_const = a_0_eq_h.lhs.subs(h(z),0)
_sc = -1/(d[5]*lamda[1]**2)
ham_h = Eq(
    _sc*_a0h_const, 
    sum(_sc*_ for _ in (a_0_eq_h.rhs - a_0_eq_h.lhs + _a0h_const).args)
)
ham_u_v_hat = ham_h.subs([h_u_v_hat.args, (gamma[2], -gamma[1])])
ham_h
h_u_v_hat
ham_u_v_hat

# uvhat12_sum_to_uv12_sum
# uv_to_uvhat_sum
# uv_j_to_uvhat_j_sym
# h_u_v_hat
# h_as_u_v
# h_as_rhohat
# rhohat_as_h

In [ ]:
d_u_v_h_hats_from_ham = [
    Eq(
        diff(uhat(z, mu[1]),z), 
        diff(ham_u_v_hat.rhs, vhat(z, mu[1]))
        .expand().collect(three_term_products + _u_v_hats, factor)
    ),
    Eq(
        diff(uhat(z, mu[2]),z), 
        diff(ham_u_v_hat.rhs, vhat(z, mu[2]))
        .expand().collect(three_term_products + _u_v_hats, factor)
    ),
    Eq(
        diff(vhat(z, mu[1]),z), 
        -diff(ham_u_v_hat.rhs, uhat(z, mu[1]))
        .expand().collect(three_term_products + _u_v_hats, factor)
    ),
    Eq(
        diff(vhat(z, mu[2]),z), 
        -diff(ham_u_v_hat.rhs, uhat(z, mu[2]))
        .expand().collect(three_term_products + _u_v_hats, factor)
    )
]
d_u_v_h_hats_from_ham_subs = [_.args for _ in d_u_v_h_hats_from_ham]

for _ in d_u_v_h_hats_from_ham:
    print()
    _

In [ ]:
d_u_v_h_hats_c = [
    Eq(
        _.lhs,
        _.rhs
        .subs([h_u_v_hat.args, (gamma[2], -gamma[1])])
        .expand().collect(three_term_products + _u_v_hats, factor)
    )
    for _ in d_u_v_h_hats_b
]
d_u_v_h_hats_c_subs = [_.args for _ in d_u_v_h_hats_c]

for _ in d_u_v_h_hats_c:
    print()
    _

In [ ]:
uvhat12_to_uvhatj_bs = [
    Eq(
        (4*lamda[1]*uvhat12_to_uvhatj.rhs).subs(j,1).subs(gamma[2], -gamma[1]).expand(), 
        4*lamda[1]*uvhat12_to_uvhatj.lhs.subs(j,1).subs(gamma[2], -gamma[1])
    ),
    Eq(
        (4*lamda[1]*uvhat12_to_uvhatj.rhs).subs(j,2).subs(gamma[2], -gamma[1]).expand(), 
        4*lamda[1]*uvhat12_to_uvhatj.lhs.subs(j,2).subs(gamma[2], -gamma[1])
    )
]
uvhat12_to_uvhatj_bs_subs = [_.args for _ in uvhat12_to_uvhatj_bs]

duvhats_from_ham = [
    Eq(
        Derivative(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]), z),
        Derivative(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]), z)
        .doit()
        .subs(d_u_v_h_hats_from_ham_subs)
        .factor()
        .subs([uvhat12_to_uvhatj_bs[_j].args])
    )
    for _j in range(2)
]

duvhats_from_c = [
    Eq(
        Derivative(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]), z),
        Derivative(uhat(z, mu[_j+1])*vhat(z, mu[_j+1]), z)
        .doit()
        .subs(d_u_v_h_hats_c_subs)
        .factor()
        .subs(uvhat12_to_uvhatj_bs_subs)
    )
    for _j in range(2)
]


assert (
    (duvhats_from_ham[0].rhs - duvhats_from_ham[1].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .simplify()
) == 0, 'intermodal power not conserved from ham'
assert (
    (duvhats_from_c[0].rhs - duvhats_from_c[1].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .simplify()
) == 0, 'intermodal power not conserved from c'

assert (
    (duvhats_from_ham[0].rhs/duvhats_from_c[0].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .factor()
) == 1, 'ham not giving correct mode power diff for mode 1' 
assert (
    (duvhats_from_ham[1].rhs/duvhats_from_c[1].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .factor()
) == 1, 'ham not giving correct mode power diff for mode 2' 

for _ in duvhats_from_ham:
    _
    
for _ in duvhats_from_c:
    _


In [ ]:
h_as_u_v
uvhat_j_rho
h_as_rhohat
rhohat_as_h

In [ ]:
uhat_sigma_j
vhat_sigma_j
uvhat_wp_j
uv_j_d5_gam_wp_z1
uv_12_d5_gam_wp_z1
uvhat12_to_uvhatj
uv_j_to_uvhat_j
uvhat_integral_log_sigma
uvhat_integral_phiz

In [ ]:
d_u_v_h_hats_c[0].rhs
print()
d_u_v_h_hats_from_ham[0].rhs

In [ ]:
_eq1 = (
    (d_u_v_h_hats_c[0].rhs - d_u_v_h_hats_from_ham[0].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .expand()
    .collect(
        three_term_products + [rhohat(z)*_ for _ in _u_v_hats] + _u_v_hats, factor
    )
)
_eq2 = (
    (d_u_v_h_hats_c[1].rhs - d_u_v_h_hats_from_ham[1].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .expand()
    .collect(
        three_term_products + [rhohat(z)*_ for _ in _u_v_hats] + _u_v_hats, factor
    )
)

_eq1
print()
_eq2

In [ ]:
_xx = Eq(
    a[1,1]*lamda[1] - a[2,2]*lamda[1] + b[2]*gamma[1],
    x
)

_xx

In [ ]:
b_j_simpd = [_.doit().subs(gams2_to_1_subs).subs(a[2,1], a[1,2]) for _ in b_j_coeffs]
b_j_simpd_subs = [_.args for _ in b_j_simpd]

for _ in b_j_simpd:
    _

In [ ]:
solve(_xx.subs([b_j_coeffs[2].doit().subs(gams2_to_1_subs).subs(a[2,1], a[1,2]).args]).simplify().subs(x,0),a[1,2])

In [ ]:

(
    gamma_lamda_bj_gam1.rhs
    .subs(b_j_simpd_subs)
    .subs(a[1,2], (-(a[1, 1] + a[2, 2])*gamma[1]/2 - a[1, 1]*lamda[1] + a[2, 2]*lamda[1])/gamma[1])
    .simplify()
)

In [ ]:
gamma_lamda_bj_gam1 
d4_lam_0

In [ ]:
(
    _eq1
    .subs([
        (a[1,1], (C - ( - a[2,2]*lamda[1] + b[2]*gamma[1]))/lamda[1]),
    ])
    .expand()
    .collect(
        three_term_products + [rhohat(z)*_ for _ in _u_v_hats] + _u_v_hats, factor
    )
    .subs([gamma_lamda_bj_gam1.args])
    .expand()
    .collect(
        three_term_products + [rhohat(z)*_ for _ in _u_v_hats] + _u_v_hats, factor
    )
)

In [ ]:
(
    (d_u_v_h_hats_c[0].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .expand()
    .collect(
        three_term_products + [rhohat(z)*_ for _ in _u_v_hats] + _u_v_hats, factor
    )
)
print()
(
    (d_u_v_h_hats_from_ham[0].rhs)
    .subs(uv_hats_to_rhohat_subs)
    .subs(gams2_to_1_subs)
    .subs([gammahat_1_to_gamma_1.args])
    .expand()
    .collect(
        three_term_products + [rhohat(z)*_ for _ in _u_v_hats] + _u_v_hats, factor
    )
)

In [ ]:
We 